In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("test.csv")

In [4]:
df.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [5]:
df.shape

(418, 11)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Pclass       418 non-null    int64  
 2   Name         418 non-null    object 
 3   Sex          418 non-null    object 
 4   Age          332 non-null    float64
 5   SibSp        418 non-null    int64  
 6   Parch        418 non-null    int64  
 7   Ticket       418 non-null    object 
 8   Fare         417 non-null    float64
 9   Cabin        91 non-null     object 
 10  Embarked     418 non-null    object 
dtypes: float64(2), int64(4), object(5)
memory usage: 36.1+ KB


In [7]:
df.isnull().sum()

PassengerId      0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64

In [8]:
df.isnull().sum().sum()

np.int64(414)

In [9]:
df.duplicated().sum()

np.int64(0)

In [10]:
df[df.duplicated()]

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked


In [11]:
df.dtypes

PassengerId      int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object

In [15]:
df.describe()

,PassengerId,Pclass,Age,SibSp,Parch,Fare
count,418.000000,418.000000,332.000000,418.000000,418.000000,417.000000
mean,1100.500000,2.265550,30.272590,0.447368,0.392344,35.627188
std,120.810458,0.841838,14.181209,0.896760,0.981429,55.907576
min,892.000000,1.000000,0.170000,0.000000,0.000000,0.000000
25%,996.250000,1.000000,21.000000,0.000000,0.000000,7.895800
50%,1100.500000,3.000000,27.000000,0.000000,0.000000,14.454200
75%,1204.750000,3.000000,39.000000,1.000000,0.000000,31.500000
max,1309.000000,3.000000,76.000000,8.000000,9.000000,512.329200


In [16]:
print("Sex values:")
print(df["Sex"].unique())

print("\nEmbarked values:")
print(df["Embarked"].unique())

print("\nPclass values:")
print(df["Pclass"].unique())

Sex values:
['male' 'female']

Embarked values:
['Q' 'S' 'C']

Pclass values:
[3 2 1]


In [17]:
print("Invalid Pclass:", df[~df["Pclass"].isin([1, 2, 3])].shape[0])
print("Invalid Age:", df[(df["Age"] < 0) | (df["Age"] > 100)].shape[0])
print("Invalid Fare:", df[df["Fare"] < 0].shape[0])
print("Invalid SibSp:", df[df["SibSp"] < 0].shape[0])
print("Invalid Parch:", df[df["Parch"] < 0].shape[0])

Invalid Pclass: 0
Invalid Age: 0
Invalid Fare: 0
Invalid SibSp: 0
Invalid Parch: 0


### Missing Data Handling

- **Age:** Missing values will be replaced using the median because Age is a numerical variable and the median is less affected by extreme values than the mean.

- **Fare:** Only one value is missing, so it will be replaced using the median Fare. This avoids deleting an otherwise useful passenger record.

- **Cabin:** A large number of Cabin values are missing. Since filling these values with an estimated cabin number would create misleading information, the missing values will be retained as "Unknown". This preserves the records while clearly indicating that the cabin information was unavailable.

These strategies were selected based on the type of variable, the amount of missing data, and the potential impact of imputation on the accuracy of the dataset.

In [18]:
df["Age"] = df["Age"].fillna(df["Age"].median())

df["Fare"] = df["Fare"].fillna(df["Fare"].median())

df["Cabin"] = df["Cabin"].fillna("Unknown")

In [19]:
df.isnull().sum()

PassengerId    0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Cabin          0
Embarked       0
dtype: int64

### Duplicate Removal

The dataset was checked for duplicate rows before cleaning. No duplicate rows were found, so no records were removed during this step.

**Number of duplicates removed: 0**

In [20]:
duplicate_count = df.duplicated().sum()
print("Duplicate rows:", duplicate_count)

df = df.drop_duplicates()

print("Rows after duplicate removal:", len(df))

Duplicate rows: 0
Rows after duplicate removal: 418


### Standardisation

The categorical values in `Sex` and `Embarked` were inspected and found to be consistent. No value replacements were necessary.

Text-based columns were standardised by removing unnecessary leading and trailing whitespace to ensure consistent formatting.

In [21]:
text_columns = ["Name", "Sex", "Ticket", "Cabin", "Embarked"]

for col in text_columns:
    df[col] = df[col].str.strip()

### Data Type Correction

The data types were reviewed to ensure that each column uses an appropriate format. Text-based columns are converted to the `string` dtype, while numerical columns such as `Pclass`, `Age`, `SibSp`, `Parch`, and `Fare` retain their numerical types. `PassengerId` is retained as an integer because it is a numeric identifier.

In [22]:
text_columns = ["Name", "Sex", "Ticket", "Cabin", "Embarked"]

for col in text_columns:
    df[col] = df[col].astype("string")

In [23]:
df.dtypes

PassengerId             int64
Pclass                  int64
Name           string[python]
Sex            string[python]
Age                   float64
SibSp                   int64
Parch                   int64
Ticket         string[python]
Fare                  float64
Cabin          string[python]
Embarked       string[python]
dtype: object

### Outlier Detection

The Interquartile Range (IQR) method is used to identify potential outliers in numerical columns. Values below Q1 − 1.5 × IQR or above Q3 + 1.5 × IQR are considered potential outliers.

Outliers will be examined before deciding whether they should be removed, capped, or retained. A value will not be removed automatically because an unusual value is not necessarily an incorrect value.

In [24]:
numeric_columns = ["Age", "SibSp", "Parch", "Fare"]

for col in numeric_columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]

    print(f"{col}: {len(outliers)} potential outliers")

Age: 36 potential outliers
SibSp: 11 potential outliers
Parch: 94 potential outliers
Fare: 55 potential outliers


In [25]:
for col in numeric_columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    print(f"\n{col}")
    print(f"Lower bound: {lower_bound:.2f}")
    print(f"Upper bound: {upper_bound:.2f}")


Age
Lower bound: 3.88
Upper bound: 54.88

SibSp
Lower bound: -1.50
Upper bound: 2.50

Parch
Lower bound: 0.00
Upper bound: 0.00

Fare
Lower bound: -27.47
Upper bound: 66.84


### Outlier Treatment Decision

The IQR method identified potential outliers in `Age`, `SibSp`, `Parch`, and `Fare`.

After examining the results, the identified values were retained rather than removed or capped. These values are plausible observations within the context of the Titanic dataset. For example, older passengers, larger family groups, and higher ticket fares are legitimate possibilities and do not indicate data-entry errors.

Therefore, no numerical values were removed or modified based solely on the IQR method. This prevents the loss of valid information from the dataset.

### Before vs. After Cleaning Summary

The following table compares the dataset before and after the cleaning process.

In [26]:
summary = pd.DataFrame({
    "Metric": [
        "Number of Rows",
        "Number of Missing Values",
        "Number of Duplicate Rows",
        "Number of Columns"
    ],
    "Before Cleaning": [
        418,
        414,
        0,
        11
    ],
    "After Cleaning": [
        len(df),
        df.isnull().sum().sum(),
        df.duplicated().sum(),
        len(df.columns)
    ]
})

summary

,Metric,Before Cleaning,After Cleaning
0,Number of Rows,418,418
1,Number of Missing Values,414,0
2,Number of Duplicate Rows,0,0
3,Number of Columns,11,11


In [27]:
df.dtypes

PassengerId             int64
Pclass                  int64
Name           string[python]
Sex            string[python]
Age                   float64
SibSp                   int64
Parch                   int64
Ticket         string[python]
Fare                  float64
Cabin          string[python]
Embarked       string[python]
dtype: object

In [28]:
df.to_csv("cleaned_titanic.csv", index=False)

In [29]:
import os
print(os.path.exists("cleaned_titanic.csv"))

True


In [30]:
df.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,Unknown,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,Unknown,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,Unknown,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,Unknown,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,Unknown,S


## Conclusion

The Titanic dataset was systematically cleaned and prepared for analysis. Missing values were handled using appropriate strategies, duplicate rows were checked and removed where necessary, text data was standardised, data types were corrected, and potential outliers were identified using the IQR method.

The final dataset contains 418 rows and 11 columns with no missing values or duplicate rows. Legitimate extreme values were retained to avoid removing valid information. The cleaned dataset was saved as `cleaned_titanic.csv` for further analysis.